# Aufbau Wissensgraph mit RDF-Lib

In [1]:
import pandas as pd
import kagglehub
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

path = kagglehub.dataset_download("nelgiriyewithana/world-stock-prices-daily-updating")

print("Path to dataset files:", path)
data_path = path+"\World-Stock-Prices-Dataset.csv"
data = pd.read_csv(data_path)

<>:9: SyntaxWarning: invalid escape sequence '\W'
<>:9: SyntaxWarning: invalid escape sequence '\W'
C:\Users\s3phi\AppData\Local\Temp\ipykernel_31872\2321265967.py:9: SyntaxWarning: invalid escape sequence '\W'
  data_path = path+"\World-Stock-Prices-Dataset.csv"
c:\workspace\StockPredictor\StockPredictor_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\s3phi\.cache\kagglehub\datasets\nelgiriyewithana\world-stock-prices-daily-updating\versions\400


In [2]:
brands = []
arr = data["Brand_Name"].unique()
for i in arr:
    brands.append(i.replace(" ", "-"))

brands

['peloton',
 'crocs',
 'adidas',
 'amazon',
 'apple',
 'nike',
 'target',
 'google',
 'spotify',
 'zoom-video-communications',
 'the-walt-disney-company',
 'roblox',
 'delta-air-lines',
 'costco',
 'southwest-airlines',
 'american-eagle-outfitters',
 'tesla',
 'starbucks',
 'nvidia',
 'salesforce-/-slack',
 'honda',
 'colgate-palmolive',
 'hershey-company',
 'chipotle',
 'pinterest',
 'logitech',
 'shopify',
 'amd',
 'american-express',
 'coinbase',
 'mastercard',
 "mcdonald's",
 'adobe',
 'unilever',
 'cisco',
 'jpmorgan-chase-&-co',
 'airbnb',
 'marriott',
 'toyota',
 'hilton',
 'the-home-depot',
 'johnson-&-johnson',
 'uber',
 'procter-&-gamble',
 'fedex',
 '3m',
 'philips',
 'foot-locker',
 'the-coca-cola-company',
 'microsoft',
 'visa',
 'lvmh',
 'zoominfo',
 'ubisoft',
 'netflix',
 'puma',
 'nintendo',
 'bmw-group',
 'porsche',
 'twitter',
 'nordstrom',
 'block']

In [63]:
df_brands = pd.DataFrame({"brands" : brands})
wikidata_elements = ["Q56276186", "Q926699", "Q3895", "Q3884", "Q312", "Q483915", "Q1046951", "Q95", "Q689141", "Q17460900", "Q7414", "Q67186598", "Q188920", "Q715583", "Q503308",
    "Q2842931", "Q478214", "Q37158", "Q182477", "Q941127", "Q9584", "Q609466", "Q868666", "Q465751", "Q96095585", "Q223127", "Q7501150", "Q128896", "Q194360",
    "Q16972754", "Q489921", "Q38076", "Q11463", "Q157062", "Q173395", "Q192314", "Q63327", "Q1141173", "Q53268", "Q1057464", "Q864407", "Q333718", "Q780442",
    "Q212405", "Q459477", "Q159433", "Q170416", "Q63335", "Q3295867", "Q2283", "Q328840", "Q504998", "Q8074134", "Q188273", "Q907311", "Q157064", "Q8093", "Q26678", "Q40993", "Q918", "Q174310", "Q30258651"]
df_brands["wikidata"] = wikidata_elements
df_brands.head(33)

,brands,wikidata
0,peloton,Q56276186
1,crocs,Q926699
2,adidas,Q3895
3,amazon,Q3884
4,apple,Q312
5,nike,Q483915
6,target,Q1046951
7,google,Q95
8,spotify,Q689141
9,zoom-video-communications,Q17460900


In [104]:
company = df_brands.loc[df_brands["brands"] == "peloton"]
brand_identifier = company["wikidata"].loc[company.index[0]]
print(brand_identifier)


Q56276186


In [77]:
query_start = "SELECT ?officialname ?logo ?inception ?totalassets ?revenue ?netprofit ?operatingincome ?marketcapitalization\n" \
"WHERE {"

query_order = "wd:"+brand_identifier+" wdt:P1448 ?officialname;\n wdt:P154 ?logo;\n wdt:P571 ?inception;\n wdt:P2403 ?totalassets;\n wdt:P2139 ?revenue;\n wdt:P2295 ?netprofit;\n wdt:P3362 ?operatingincome;\n wdt:P2226 ?marketcapitalization.\n"

query_end = "}"

query1 = query_start+query_order+query_end

print(query1)

SELECT ?officialname ?logo ?inception ?totalassets ?revenue ?netprofit ?operatingincome ?marketcapitalization
WHERE {wd:Q328840 wdt:P1448 ?officialname;
 wdt:P154 ?logo;
 wdt:P571 ?inception;
 wdt:P2403 ?totalassets;
 wdt:P2139 ?revenue;
 wdt:P2295 ?netprofit;
 wdt:P3362 ?operatingincome;
 wdt:P2226 ?marketcapitalization.
}


In [109]:
query_head = "SELECT ?official ?logo ?inception ?totalassets ?revenue ?netprofit ?operatingincome ?marketcapitalization\n" \
"WHERE {"
query_order = "OPTIONAL { wd:"+brand_identifier+" wdt:P1448 ?officialname } \nOPTIONAL { wd:"+brand_identifier+" wdt:P856 ?officialwebsite } \nBIND(IF(BOUND(?officialname), ?officialname, ?officialwebsite) AS ?official) \nOPTIONAL { wd:"+brand_identifier+" wdt:P154 ?logo } \nOPTIONAL { wd:"+brand_identifier+" wdt:P571 ?inception } \
    \nOPTIONAL { wd:"+brand_identifier+" wdt:P2403 ?totalassets } \
    \nOPTIONAL { wd:"+brand_identifier+" wdt:P2139 ?revenue } \
    \nOPTIONAL { wd:"+brand_identifier+" wdt:P2295 ?netprofit } \
    \nOPTIONAL { wd:"+brand_identifier+" wdt:P3362 ?operatingincome } \
    \nOPTIONAL { wd:"+brand_identifier+" wdt:P2226 ?marketcapitalization }\n}"
query1 = query_head+query_order

print(query1)

SELECT ?official ?logo ?inception ?totalassets ?revenue ?netprofit ?operatingincome ?marketcapitalization
WHERE {OPTIONAL { wd:Q56276186 wdt:P1448 ?officialname } 
OPTIONAL { wd:Q56276186 wdt:P856 ?officialwebsite } 
BIND(IF(BOUND(?officialname), ?officialname, ?officialwebsite) AS ?official) 
OPTIONAL { wd:Q56276186 wdt:P154 ?logo } 
OPTIONAL { wd:Q56276186 wdt:P571 ?inception }     
OPTIONAL { wd:Q56276186 wdt:P2403 ?totalassets }     
OPTIONAL { wd:Q56276186 wdt:P2139 ?revenue }     
OPTIONAL { wd:Q56276186 wdt:P2295 ?netprofit }     
OPTIONAL { wd:Q56276186 wdt:P3362 ?operatingincome }     
OPTIONAL { wd:Q56276186 wdt:P2226 ?marketcapitalization }
}


In [106]:
query2 = "SELECT ?inception WHERE {wd:Q1057464 wdt:P571 ?inception.}"

In [110]:
url = 'https://query.wikidata.org/sparql'
user_agent = 'Visual Studio Code/1.102.0 (Windows_NT x64 10.0.19045; timucin.cicek@stud.h-da.de)'
sparql = SPARQLWrapper(url, agent = user_agent )
#results = sparql.query()

In [111]:
#sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
# From https://www.wikidata.org/wiki/Wikidata:SPARQL_query_service/queries/examples#Cats
sparql.setQuery(query1)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

In [112]:
results

{'head': {'vars': ['official',
   'logo',
   'inception',
   'totalassets',
   'revenue',
   'netprofit',
   'operatingincome',
   'marketcapitalization']},
 'results': {'bindings': [{'official': {'type': 'uri',
     'value': 'https://www.onepeloton.com'},
    'logo': {'type': 'uri',
     'value': 'http://commons.wikimedia.org/wiki/Special:FilePath/Peloton%20%28Unternehmen%29%20logo.svg'},
    'inception': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime',
     'type': 'literal',
     'value': '2012-01-01T00:00:00Z'}}]}}

In [101]:
results

{'head': {'vars': ['official',
   'logo',
   'inception',
   'totalassets',
   'revenue',
   'netprofit',
   'operatingincome',
   'marketcapitalization']},
 'results': {'bindings': [{'official': {'xml:lang': 'de',
     'type': 'literal',
     'value': 'Bayerische Motoren Werke AG'},
    'logo': {'type': 'uri',
     'value': 'http://commons.wikimedia.org/wiki/Special:FilePath/BMW%20logo%20%28gray%29.svg'},
    'inception': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime',
     'type': 'literal',
     'value': '1916-03-07T00:00:00Z'},
    'totalassets': {'datatype': 'http://www.w3.org/2001/XMLSchema#decimal',
     'type': 'literal',
     'value': '228034000000'},
    'revenue': {'datatype': 'http://www.w3.org/2001/XMLSchema#decimal',
     'type': 'literal',
     'value': '142610000000'},
    'netprofit': {'datatype': 'http://www.w3.org/2001/XMLSchema#decimal',
     'type': 'literal',
     'value': '7680000000'},
    'operatingincome': {'datatype': 'http://www.w3.org/2001/XMLSche

In [113]:
print(results['results']['bindings'][0]["official"]["value"])
print(results['results']['bindings'][0]["inception"]["value"])
print(results['results']['bindings'][0]["totalassets"]["value"],"€")
print(results['results']['bindings'][0]["revenue"]["value"],"€")
print(results['results']['bindings'][0]["netprofit"]["value"],"€")
print(results['results']['bindings'][0]["operatingincome"]["value"],"€")
print(results['results']['bindings'][0]["marketcapitalization"]["value"],"€")
n = int(results['results']['bindings'][0]["marketcapitalization"]["value"])
res = "{:,}".format(n)
print(res,"€")

https://www.onepeloton.com
2012-01-01T00:00:00Z


KeyError: 'totalassets'

In [ ]:
print(results['results']['bindings'][0]["official"]["value"])
print(results['results']['bindings'][0]["inception"]["value"])
if results['results']['bindings'][0]["totalassets"]["value"] is NULL: #Hier noch checken, wie man prüfen kann, ob ein Wert in der SPARQL-Rückgabe vorhanden ist
    print("Es gibt keinen Wert")
print(results['results']['bindings'][0]["revenue"]["value"],"€")
print(results['results']['bindings'][0]["netprofit"]["value"],"€")
print(results['results']['bindings'][0]["operatingincome"]["value"],"€")
print(results['results']['bindings'][0]["marketcapitalization"]["value"],"€")
n = int(results['results']['bindings'][0]["marketcapitalization"]["value"])
res = "{:,}".format(n)
print(res,"€")

https://www.onepeloton.com
2012-01-01T00:00:00Z


KeyError: 'totalassets'

In [79]:
results_df = pd.DataFrame(results['results']['bindings'])
results_df

,officialname,logo,inception,totalassets,revenue,netprofit,operatingincome,marketcapitalization
0,"{'xml:lang': 'de', 'type': 'literal', 'value':...","{'type': 'uri', 'value': 'http://commons.wikim...",{'datatype': 'http://www.w3.org/2001/XMLSchema...,{'datatype': 'http://www.w3.org/2001/XMLSchema...,{'datatype': 'http://www.w3.org/2001/XMLSchema...,{'datatype': 'http://www.w3.org/2001/XMLSchema...,{'datatype': 'http://www.w3.org/2001/XMLSchema...,{'datatype': 'http://www.w3.org/2001/XMLSchema...


<div class="alert alert-block alert-info">
<b>Zusammenfassung für Gradio</b> 
</div>

In [175]:
import pandas as pd
import kagglehub
from SPARQLWrapper import SPARQLWrapper, JSON

path = kagglehub.dataset_download("nelgiriyewithana/world-stock-prices-daily-updating")

print("Path to dataset files:", path)
data_path = path+"\World-Stock-Prices-Dataset.csv"
data = pd.read_csv(data_path)

<>:8: SyntaxWarning: invalid escape sequence '\W'
<>:8: SyntaxWarning: invalid escape sequence '\W'
C:\Users\s3phi\AppData\Local\Temp\ipykernel_16436\1466582878.py:8: SyntaxWarning: invalid escape sequence '\W'
  data_path = path+"\World-Stock-Prices-Dataset.csv"


Path to dataset files: C:\Users\s3phi\.cache\kagglehub\datasets\nelgiriyewithana\world-stock-prices-daily-updating\versions\391


In [196]:
brands = []
arr = data["Brand_Name"].unique() #gebe jede firma ohne dubletten an
for i in arr:
    brands.append(i.replace(" ", "-")) #lösche leerzeichen und ersetze diese durch bindestriche

df_brands = pd.DataFrame({"brands" : brands}) #neues data frame erstellen mir den firmen
wikidata_elements = ["Q56276186", "Q926699", "Q3295867", "Q3895", "Q194360", "Q157064", "Q328840", "Q11463", "Q157062", "Q173395", "Q192314", "Q504998", "Q63327", "Q1141173", "Q8074134", "Q53268", "Q1057464", 
                     "Q38076", "Q864407", "Q489921", "Q333718", "Q780442", "Q212405", "Q16972754", "Q459477", "Q159433", "Q170416", "Q63335", "Q907311", "Q128896", "Q188273", "Q7501150", "Q503308", "Q223127", 
                     "Q3884", "Q312", "Q483915", "Q1046951", "Q95", "Q689141", "Q17460900", "Q7414", "Q67186598", "Q8093", "Q188920", "Q2283", "Q715583", "Q2842931", "Q609466", "Q96095585", "Q26678", 
                     "Q465751", "Q868666", "Q40993", "Q9584", "Q941127", "Q182477", "Q37158", "Q478214", "Q918", "Q174310", "Q30258651"]
df_brands["wikidata"] = wikidata_elements #füge die wikidata identifier als neue spalte hinzu

company = df_brands.loc[df_brands["brands"] == "bmw-group"]
brand_identifier = company["wikidata"].loc[company.index[0]]

query_start = "SELECT ?officialname ?logo ?inception ?totalassets ?revenue ?netprofit ?operatingincome ?marketcapitalization\n WHERE {"
query_order = "wd:"+brand_identifier+" wdt:P1448 ?officialname;\n wdt:P154 ?logo;\n wdt:P571 ?inception;\n wdt:P2403 ?totalassets;\n wdt:P2139 ?revenue;\n wdt:P2295 ?netprofit;\n wdt:P3362 ?operatingincome;\n wdt:P2226 ?marketcapitalization.\n"
query_end = "}"

query1 = query_start+query_order+query_end

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
# From https://www.wikidata.org/wiki/Wikidata:SPARQL_query_service/queries/examples#Cats
sparql.setQuery(query1)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

print(results['results']['bindings'][0]["officialname"]["value"])
print(results['results']['bindings'][0]["logo"]["value"])
print(results['results']['bindings'][0]["inception"]["value"])
print(results['results']['bindings'][0]["totalassets"]["value"],"€")
print(results['results']['bindings'][0]["revenue"]["value"],"€")
print(results['results']['bindings'][0]["netprofit"]["value"],"€")
print(results['results']['bindings'][0]["operatingincome"]["value"],"€")
print(results['results']['bindings'][0]["marketcapitalization"]["value"],"€")
marketcaptl = "{:,}".format(int(results['results']['bindings'][0]["marketcapitalization"]["value"]))+" €"
print(marketcaptl)

Bayerische Motoren Werke AG
http://commons.wikimedia.org/wiki/Special:FilePath/BMW%20logo%20%28gray%29.svg
1916-03-07T00:00:00Z
228034000000 €
142610000000 €
7680000000 €
13999000000 €
59907000000 €
59,907,000,000 €


In [195]:
inception = results['results']['bindings'][0]["inception"]["value"]
inception_short = inception[:10]
inception_short

'2003-07-01'